In [ ]:
!unzip /content/model_latest.zip
!rm -rf /content/model_latest.zip

Archive:  /content/model_latest.zip
 extracting: model_latest/config.json  
 extracting: model_latest/model.safetensors  
 extracting: model_latest/tokenizer.json  
 extracting: model_latest/tokenizer_config.json  


In [ ]:
!pip install onnxsim

In [ ]:
!pip install onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 9.2 MB/s eta 0:00:00


In [ ]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import onnx
from onnxsim import simplify
from onnxruntime.quantization import quantize_dynamic, QuantType

print("==================================================")
print(" INICIANDO CONVERSÃO NATIVA COM ONNX-SIMPLIFIER")
print("==================================================\n")

caminho_modelo = "./model_latest"
pasta_saida = "./modelo_onnx"

os.makedirs(pasta_saida, exist_ok=True)

# 1. Carregamento
print("1. Carregando modelo via PyTorch...")
tokenizer = AutoTokenizer.from_pretrained(caminho_modelo)
model = AutoModelForSequenceClassification.from_pretrained(caminho_modelo)

model.eval()
tokenizer.save_pretrained(pasta_saida)

texto_exemplo = "Esta é uma mensagem de teste para o ONNX mapear os tensores."
inputs = tokenizer(texto_exemplo, return_tensors="pt", max_length=128, padding="max_length", truncation=True)

caminho_onnx_base = f"{pasta_saida}/model_base.onnx"
caminho_onnx_sim = f"{pasta_saida}/model_sim.onnx"
caminho_onnx_quantizado = f"{pasta_saida}/model_quantized.onnx"

# 2. Exportação Direta via PyTorch
print("2. Exportando o grafo matemático bruto para ONNX...")
with torch.no_grad():
    torch.onnx.export(
        model,
        (inputs['input_ids'], inputs['attention_mask']),
        caminho_onnx_base,
        input_names=['input_ids', 'attention_mask'],
        output_names=['logits'],
        dynamic_axes={
            'input_ids': {0: 'batch_size', 1: 'sequence_length'},
            'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
            'logits': {0: 'batch_size'}
        },
        opset_version=14
    )

# ==============================================================
# 3. O SEGREDO: ONNX Simplifier (Resolve o erro 768 vs 2)
# ==============================================================
print("3. Simplificando o Grafo (Limpando as dimensões confusas)...")
model_onnx = onnx.load(caminho_onnx_base)
model_simp, check = simplify(model_onnx)
assert check, "A simplificação falhou!"
onnx.save(model_simp, caminho_onnx_sim)
print("   ✓ Grafo limpo e corrigido!")

# 4. Quantização (Compressão para 8-bits)
print("4. Aplicando Quantização Dinâmica (INT8)...")
quantize_dynamic(
    model_input=caminho_onnx_sim,           # Lemos o arquivo SIMPLIFICADO agora!
    model_output=caminho_onnx_quantizado,
    weight_type=QuantType.QInt8
)

print("\n==================================================")
print("✅ SUCESSO! Modelo convertido e quantizado perfeitamente.")
print("==================================================")
print(f"O seu arquivo 'model_quantized.onnx' está pronto na pasta: {pasta_saida}")

 INICIANDO CONVERSÃO NATIVA COM ONNX-SIMPLIFIER

1. Carregando modelo via PyTorch...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2. Exportando o grafo matemático bruto para ONNX...


/tmp/ipykernel_3362/2412585032.py:35: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0930 19:13:22.772000 3362 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `BertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


... (2 outputs left out)

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: sequence_length will not be used, since it shares the same shape constraints with another axis: sequence_length.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


3. Simplificando o Grafo (Limpando as dimensões confusas)...
   ✓ Grafo limpo e corrigido!
4. Aplicando Quantização Dinâmica (INT8)...



✅ SUCESSO! Modelo convertido e quantizado perfeitamente.
O seu arquivo 'model_quantized.onnx' está pronto na pasta: ./modelo_onnx


In [ ]:
import onnxruntime as ort
from transformers import AutoTokenizer
import numpy as np

# =====================================================================
# ESCOPO GLOBAL (Carregado apenas 1x durante o Cold Start do Lambda)
# =====================================================================
CAMINHO_MODELO = "./modelo_onnx"

# Configurações para extrair a máxima performance da CPU da AWS
opcoes_sessao = ort.SessionOptions()
opcoes_sessao.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# Carregamento do Tokenizador e do Modelo Quantizado
tokenizer = AutoTokenizer.from_pretrained(CAMINHO_MODELO)
sessao_onnx = ort.InferenceSession(
    f"{CAMINHO_MODELO}/model_quantized.onnx",
    sess=opcoes_sessao,
    providers=["CPUExecutionProvider"]
)

# =====================================================================
# FUNÇÃO EXPORTÁVEL (Executada a cada requisição)
# =====================================================================
def analisar_fraude(mensagem: str) -> dict:
    """
    Recebe uma string, processa via ONNX Runtime e retorna o veredito.
    """
    if not mensagem or not mensagem.strip():
        return {"is_phishing": False, "label": 0, "confianca": 0.0, "erro": "Mensagem vazia"}

    try:
        # 1. Tokenização (Transforma o texto em números para a matriz ONNX)
        inputs = tokenizer(
            mensagem,
            return_tensors="np",  # NumPy arrays, exigência do ONNX
            padding="max_length",
            truncation=True,
            max_length=128
        )

        # Filtra apenas as chaves que a arquitetura do modelo espera
        entradas_onnx = {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"]
        }

        # 2. Inferência C++ (Retorna os 'logits' brutos do modelo)
        saidas = sessao_onnx.run(None, entradas_onnx)
        logits = saidas[0][0]

        # 3. Decodificação (Calcula a classe vencedora e a porcentagem de certeza)
        label_previsto = int(np.argmax(logits))

        # Cálculo de Softmax manual leve para evitar importar bibliotecas pesadas de matemática
        exp_logits = np.exp(logits - np.max(logits))
        probabilidades = exp_logits / exp_logits.sum()
        confianca = float(probabilidades[label_previsto])

        return {
            "is_phishing": bool(label_previsto == 1),
            "label": label_previsto,
            "confianca": round(confianca, 4)
        }

    except Exception as e:
        # Log seguro para não derrubar a API caso ocorra um erro de codificação bizarra
        return {"is_phishing": False, "label": 0, "confianca": 0.0, "erro": str(e)}

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
# Se estiver em arquivos separados, descomente a linha abaixo:
# from predictor import analisar_fraude

print("==================================================")
print(" INICIANDO AVALIAÇÃO DE DESEMPENHO (ONNX)")
print("==================================================\n")

# 1. A sua lista de testes rotulada (Ground Truth)
mensagens_teste = [
    # --- MENSAGENS NORMAIS (Label 0) ---
    ("oi, n esquece de passar na padaria voltando do trampo", 0),
    ("a reunião mudou pra sala 3, avisa a glr pfv", 0),
    ("vc viu o jogo ontem? q golaço no final slc!", 0),
    ("seu pacote do magalu chegou aqui na portaria, dps pega la", 0),
    ("feliz natal fml!! q a noite de vcs seja top", 0),
    ("preciso q vc me mande o pdf daquela apresentaçao ate umas 12h", 0),
    ("to no transito, vou atrasar uns 10 min malz", 0),
    ("bom dia! o conserto ficou 150 conto. pd fzr o pix no meu cel msm", 0),
    ("oie, passando p/ lembrar da consulta cm o dr marcos amanhã as 15h blz?", 0),
    ("bora marcar um fut esse fds?", 0),
    ("amor, a raçao do dog acabou, compra + dps", 0),
    ("a fatura da vivo ja ta no app, n esquece de pagar", 0),
    ("mae cheguei dboa no hotel. mais tarde te ligo bjs", 0),
    ("alguem tem ctt de um encanador bom pra me passar?", 0),
    ("a prof de ingles mandou o link da aula no gp", 0),
    ("oi, dsclp a demora pra responder, tava em reuniao", 0),
    ("vc vai almoçar em ksa hj ou vai comer fora?", 0),
    ("parabens mano!! mtas felicidades e sucesso smp tmj", 0),
    ("o uber ta chegando, 2 minutinhos", 0),
    ("o boleto do cond ja ta no seu email", 0),
    ("vou viajar no feriado, dps a gnt se ve", 0),
    ("tem pão fresco saindo agr, qr q eu leve?", 0),
    ("a vacina da gripe começou no posto aq da esquina", 0),
    ("n consegui falar c/ vc, me liga qnd der", 0),
    ("confirmando nosso jantar de amanhã as 20h. ate!", 0),

    # --- MENSAGENS DE PHISHING / FRAUDE (Label 1) ---
    ("Bacen: tem um pix de 850 conto retido no seu cpf. acessa p/ liberar: ", 1),
    ("C6 Bank: cartao bloqueado por seguranca. valida ai rapidao senao cancela: ", 1),
    ("Oii mae, meu cel quebrou :( salva meu num novo provisorio ai", 1),
    ("Opa, sou do RH da Amazon. Vaga meio periodo tirando de 200 a 500/dia. Clica ai: ", 1),
    ("Detran: suspensao de CNH agendada. Paga o debito p/ evitar BO no doc: ", 1),
    ("gov.br: sua conta vai ser desativada hj! atualiza seus dados aqui ", 1),
    ("Correios: seu pedido ficou retido na alfandega. Tem taxa de 22,50 pendente no pix: ", 1),
    ("Netflix: o pgto falhou :( atualiza o cartao pra n perder o acesso: ", 1),
    ("Promoção O Boticário bombando! responde a pesquisa e ganha 1 kit de dia das mães free: ", 1),
    ("Itau: acesso em outro estado detectado. Nao foi vc? Contesta aq no link [URL]", 1),
    ("vc tem 20k de pontos Livelo expirando HOJE!! troca ai por milha no link ", 1),
    ("Serasa: limpe seu nome c/ 95% off no feirao! paga no pix oficial aq: ", 1),
    ("Governo liberou Auxilio Gas de 120$!! ve se seu cpf tem direito [URL]", 1),
    ("nossa, vc viu as fotos suas da festa q vazaram?? q loucura, olha ai [URL] ", 1),
    ("Oie, vc foi selecionado p/ avaliar produtos na Shopee e tirar +300/dia. Chama no whats: ", 1),
    ("Nubank: compra de 2.500 aprovada no mercadopago. nao reconhece? clica aq pra cancelar: ", 1),
    ("Receita Federal: seu IRPF caiu na malha fina. acesse pra ver oq deu errado: ", 1),
    ("Suporte WhatsApp: digite o cod q vc recebeu por sms pra n perder o whats.", 1),
    ("Caixa Tem: saque extraordinario na conta!! clica pra transferir o dindin: ", 1),
    ("Santander: app desatualizado. atualiza no link p/ n tomar multa no cpf: ", 1),
    ("quer fzr renda extra?? investe 50tao no robo do pix e tira 500 na hora. clica: ", 1),
    ("Claro: fatura em aberto com 50% de desc hj no pix. gera a chave ai: ", 1),
    ("Atenção! açao judicial no seu CPF. ve o processo e o mandado aqui: ", 1),
    ("emprestimo de 10k aprovado ate pra negativado slc!! s/ consulta no spc. assina ai: ", 1),
    ("Mercado Livre: conta suspensa p/ ativ suspeita. confirma q é vc aqui ", 1),
        ("Oi, não esquece de passar na padaria voltando do trabalho.", 0),
    ("A reunião foi transferida para a sala 3, avisem os demais.", 0),
    ("Você viu o jogo ontem? Que golaço no final!", 0),
    ("Seu pedido Magalu 4920 foi entregue ao destinatário.", 0),
    ("Feliz Natal, família! Que a noite de vocês seja iluminada.", 0),
    ("Preciso que você me envie o PDF daquela apresentação até meio-dia.", 0),
    ("Estou no trânsito, devo atrasar uns 10 minutinhos.", 0),
    ("Bom dia, o valor do conserto ficou em R$ 150. Pode fazer o Pix para a chave celular.", 0),
    ("Lembrete: Consulta com o Dr. Marcos amanhã às 15:00.", 0),
    ("Bora marcar um futebol no fim de semana?", 0),
    ("Amor, a ração do cachorro acabou, compra mais depois.", 0),
    ("A sua fatura da Vivo com vencimento em 10/08 está disponível no app.", 0),
    ("Mãe, cheguei bem no hotel. Mais tarde te ligo.", 0),
    ("Alguém tem o contato de um bom encanador pra me passar?", 0),
    ("A professora de inglês mandou o link da aula no grupo.", 0),
    ("Oi, desculpa a demora pra responder, tava em reunião.", 0),
    ("Você vai almoçar em casa hoje ou no restaurante?", 0),
    ("Parabéns, amigo! Muitas felicidades e sucesso sempre.", 0),
    ("Uber: Seu motorista Carlos está chegando em 2 minutos.", 0),
    ("O boleto do condomínio já foi enviado pro seu email.", 0),
    ("Vou viajar no feriado, depois a gente se vê.", 0),
    ("Tem pão fresco saindo agora, vai querer que eu leve?", 0),
    ("A vacinação da gripe começou no posto aqui da esquina.", 0),
    ("Não consegui falar com você, me liga quando puder.", 0),
    ("Confirmando o nosso jantar de amanhã às 20h. Até lá!", 0),

    # --- MENSAGENS DE PHISHING / FRAUDE (Label 1) ---
    ("BANCO CENTRAL: Há um Pix de R$ 850 retido em seu nome. Acesse para liberar: ", 1),
    ("C6 Bank: Seu cartão foi bloqueado por segurança. Valide sua identidade imediatamente: ", 1),
    ("Mãe, meu celular quebrou, salva esse meu número novo provisório.", 1),
    ("Olá, somos do RH da Amazon. Temos vaga de meio período para ganhar de 200 a 500 por dia. Clique: ", 1),
    ("Detran informa: Suspensão de CNH agendada. Pague o débito para evitar o cancelamento: ", 1),
    ("Aviso gov.br: Sua conta será desativada em 2h. Atualize seus dados cadastrais em ", 1),
    ("CORREIOS: Seu pedido foi retido na alfândega. Taxa de R$ 22,50 pendente. Pague aqui: ", 1),
    ("Netflix: O pagamento do mês falhou. Atualize seu cartão de crédito para não perder o acesso: ", 1),
    ("Promoção O Boticário! Responda o questionário e ganhe um kit grátis de Dia das Mães: ", 1),
    ("Itaú: Identificamos um acesso no Rio de Janeiro. Caso não tenha sido você, conteste em [URL]", 1),
    ("Você tem 20.000 pontos Livelo expirando hoje! Troque por milhas ou produtos em ", 1),
    ("Serasa Consumidor: Pague sua dívida com 95% de desconto no Feirão Limpa Nome. Pix oficial: ", 1),
    ("O Governo aprovou um Auxílio Gás de R$ 120. Consulte se seu CPF tem direito: [URL]", 1),
    ("Oi, encontrei essas fotos suas da festa de ontem rolando na internet, dá uma olhada [URL] ", 1),
    ("Shopee: Parabéns, você foi selecionado para avaliar produtos. Renda diária de R$ 300. Chame no WhatsApp: ", 1),
    ("Nubank: Sua compra no valor de R$ 2.500 no MercadoPago foi aprovada. Para cancelar, clique: ", 1),
    ("Aviso Receita Federal: Sua declaração de imposto de renda caiu na malha fina. Veja a divergência: ", 1),
    ("WhatsApp: Insira o código de 6 dígitos que você recebeu por SMS para não perder sua conta.", 1),
    ("Caixa Tem: Seu saque extraordinário foi liberado. Acesse a conta para transferir o valor: ", 1),
    ("Santander: Seu aplicativo de segurança está desatualizado. Evite multas atualizando no link: ", 1),
    ("Oportunidade Única! Invista R$ 50 no nosso robô de Pix e receba R$ 500 em 10 minutos. Garanta sua vaga: ", 1),
    ("Claro: Sua fatura em aberto pode ser paga com 50% de desconto hoje no Pix. Gere a chave em: ", 1),
    ("Atenção: Ação judicial protocolada em seu CPF. Visualize o processo e mandado de intimação aqui: ", 1),
    ("Empréstimo liberado de R$ 10.000 para negativados sem consulta ao SPC! Assine o contrato online: ", 1),
    ("Mercado Livre: Seu acesso está suspenso temporariamente por atividade suspeita. Confirme sua titularidade", 1)
]

textos = [item[0] for item in mensagens_teste]
y_true = [item[1] for item in mensagens_teste]
y_pred = []

print("--- DETALHAMENTO DAS MENSAGENS ---\n")

# 2. Faz a predição chamando a sua função ONNX para cada mensagem
for texto, real in zip(textos, y_true):
    veredito = analisar_fraude(texto)

    # Extrai o label numérico (0 ou 1) da resposta do seu modelo
    label_previsto = veredito["label"]
    confianca = veredito["confianca"]
    y_pred.append(label_previsto)
    print(f"Mensagem: {texto}")
    print(f"Real: {real} | Previsto: {label_previsto} | Confiança: {confianca:.4f}\n")

# 3. Calcula as métricas matemáticas
acuracia = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='binary')

# 4. Imprime o Relatório Final
print("="*40)
print(" RESULTADOS GERAIS DA AVALIAÇÃO ONNX")
print("="*40)
print(f"Acurácia Global: {acuracia:.4f} ({acuracia * 100:.2f}%)")
print(f"F1-Score (Fraudes): {f1:.4f} ({f1 * 100:.2f}%)")

print("\nRelatório Detalhado:")
print(classification_report(y_true, y_pred, target_names=["Normal (0)", "Phishing (1)"]))

matriz = confusion_matrix(y_true, y_pred)

# Imprime a versão em texto (modo raiz)
print("Matriz de Confusão:")
print(f"Verdadeiros Negativos: {matriz[0][0]}  | Falsos Positivos: {matriz[0][1]}")
print(f"Falsos Negativos: {matriz[1][0]}       | Verdadeiros Positivos: {matriz[1][1]}")

 INICIANDO AVALIAÇÃO DE DESEMPENHO (ONNX)

--- DETALHAMENTO DAS MENSAGENS ---

Mensagem: oi, n esquece de passar na padaria voltando do trampo
Real: 0 | Previsto: 0 | Confiança: 0.9998

Mensagem: a reunião mudou pra sala 3, avisa a glr pfv
Real: 0 | Previsto: 0 | Confiança: 0.9913

Mensagem: vc viu o jogo ontem? q golaço no final slc!
Real: 0 | Previsto: 0 | Confiança: 0.9884

Mensagem: seu pacote do magalu chegou aqui na portaria, dps pega la
Real: 0 | Previsto: 0 | Confiança: 0.9951

Mensagem: feliz natal fml!! q a noite de vcs seja top
Real: 0 | Previsto: 0 | Confiança: 0.9940

Mensagem: preciso q vc me mande o pdf daquela apresentaçao ate umas 12h
Real: 0 | Previsto: 0 | Confiança: 0.9923

Mensagem: to no transito, vou atrasar uns 10 min malz
Real: 0 | Previsto: 0 | Confiança: 0.9838

Mensagem: bom dia! o conserto ficou 150 conto. pd fzr o pix no meu cel msm
Real: 0 | Previsto: 0 | Confiança: 0.9971

Mensagem: oie, passando p/ lembrar da consulta cm o dr marcos amanhã as 15h blz?
R

In [ ]:
analisar_fraude("")

{'is_phishing': True, 'label': 1, 'confianca': 0.994}